# Artifact Inventory and Legacy Quarantine

Inventories notebooks, configs, documentation, reports, checkpoints and generated output collections. Raw data filenames are deliberately excluded so the inventory cannot leak Regen PII. Classification is configuration-only; no file is deleted or moved.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data").exists(), "Could not locate project data directory"
REPO_ROOT = ROOT.parent
OUT_DIR = ROOT / "reports" / "repository_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SNAPSHOT_STARTED_UTC = datetime.now(timezone.utc).isoformat()
ALLOWED = {"authoritative", "reusable", "provisional", "legacy", "smoke-only", "generated"}

In [2]:
def repo_rel(path):
    return Path(path).relative_to(REPO_ROOT).as_posix()


def classify(path, artifact_type):
    text = repo_rel(path).lower()
    parts = set(Path(text).parts)
    name = Path(text).name

    if any(token in text for token in ("fold5", "fold_5", "fold-5", "neutral_smoke")):
        return "legacy", "forbidden fold-5 or neutral-smoke artifact"
    if "tsdf" in text or "gt_per_bone_256" in text:
        return "legacy", "legacy TSDF or unversioned target path"
    if text.endswith("/data/interim/predrr") or text.endswith("/data/interim/drrs"):
        return "legacy", "unversioned preprocessing output"
    if "smoke" in text or "test_checkpoint" in text:
        return "smoke-only", "diagnostic artifact only"
    if (
        text in {"agents.md", "claude.md"}
        or text.startswith(".codex/agents/")
        or "testproject/configs/" in text
        or "foundation-implementation-v1.md" in text
        or "quantitative_manifest_v1" in text
    ):
        return "authoritative", "foundation contract or canonical manifest"
    if "references/" in text:
        return "reusable", "immutable reference implementation"
    if artifact_type == "notebook":
        if any(token in text for token in (
            "00_manifest_and_reconciliation", "01_fracture_roi_annotation",
            "02_lps_geometry_validation", "03_notebook_environment_parity",
            "04_contract_validation", "05_vsd_merge_laterality_audit",
            "06_artifact_inventory", "07_vsd_cohort_laterality_audit", "08_vsd_focused_ct_stl_target_overlay_audit", "predrr_preprocessing", "drr_generation",
            "00_gt_per_bone", "data_augmentation",
        )):
            return "reusable", "active notebook-first foundation workflow"
        return "provisional", "not certified for final configuration"
    if artifact_type == "checkpoint":
        return "provisional", "checkpoint requires fold provenance and gate review"
    if artifact_type == "output_collection":
        if "_lps_256_v1" in text:
            return "provisional", "versioned output awaits HPC evidence"
        return "generated", "generated output collection"
    return "generated", "derived report or documentation"


records = []
def add(path, artifact_type, count=None, bytes_total=None):
    classification, reason = classify(path, artifact_type)
    records.append({
        "path": repo_rel(path),
        "artifact_type": artifact_type,
        "classification": classification,
        "reason": reason,
        "item_count": count,
        "bytes_total": bytes_total,
        "final_selectable": classification == "authoritative",
    })

for path in [REPO_ROOT / "AGENTS.md", REPO_ROOT / "CLAUDE.md"]:
    if path.exists():
        add(path, "contract")

for base in [REPO_ROOT / ".codex" / "agents", ROOT / "configs", ROOT / "docs"]:
    if base.exists():
        for path in sorted(p for p in base.rglob("*") if p.is_file()):
            add(path, "config_or_document")

for base in [ROOT / "notebooks", ROOT / "HPC" / "HPC_notebooks", ROOT / "references"]:
    if base.exists():
        for path in sorted(base.rglob("*.ipynb")):
            add(path, "notebook")

for base in [ROOT / "reports", ROOT / "HPC" / "HPC_results"]:
    if base.exists():
        for path in sorted(p for p in base.rglob("*") if p.is_file()):
            add(path, "report")

checkpoint_extensions = {".pth", ".pt", ".ckpt"}
for base in [ROOT / "artifacts", ROOT / "models", ROOT / "references"]:
    if base.exists():
        for path in base.rglob("*"):
            if path.is_file() and path.suffix.lower() in checkpoint_extensions:
                add(path, "checkpoint")

for base in [ROOT / "data" / "interim", ROOT / "data" / "processed", ROOT / "HPC" / "HPC_results"]:
    if not base.exists():
        continue
    for child in sorted(p for p in base.iterdir() if p.is_dir()):
        files = [p for p in child.rglob("*") if p.is_file()]
        add(child, "output_collection", len(files), sum(p.stat().st_size for p in files))

for expected in [
    ROOT / "data/interim/predrr_lps_256_v1",
    ROOT / "data/interim/gt_per_bone_lps_256_v1",
    ROOT / "data/interim/DRRs_diffdrr_lps_256_v1",
    ROOT / "data/interim/fracture_roi_lps_256_v1",
]:
    if not expected.exists():
        add(expected, "output_collection", 0, 0)

inventory = pd.DataFrame(records).drop_duplicates(subset=["path"]).sort_values("path").reset_index(drop=True)
assert inventory["classification"].isin(ALLOWED).all()
assert inventory["path"].is_unique
inventory.to_csv(OUT_DIR / "artifact_inventory_v1.csv", index=False)
summary = inventory.groupby(["artifact_type", "classification"]).size().rename("n").reset_index()
summary.to_csv(OUT_DIR / "artifact_inventory_summary_v1.csv", index=False)

legacy = inventory[inventory.classification.isin(["legacy", "smoke-only"])].copy()
legacy.to_csv(OUT_DIR / "legacy_quarantine_v1.csv", index=False)
policy = {
    "mode": "configuration_only_no_deletion",
    "snapshot_started_utc": SNAPSHOT_STARTED_UTC,
    "completeness_rule": "All in-scope paths present when the scan starts are classified; files created later belong to the next snapshot.",
    "unexplained_missing_at_snapshot": 0,
    "raw_data_filenames_inventoried": False,
    "regen_raw_modified": False,
    "allowed_classes": sorted(ALLOWED),
    "legacy_or_smoke_items": len(legacy),
    "final_config_forbidden": ["binary_union", "tsdf", "fold_id_5", "neutral_smoke"],
}
(OUT_DIR / "artifact_inventory_metadata_v1.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
print(summary.to_string(index=False))
print(f"inventory rows={len(inventory)}, quarantined={len(legacy)}")

     artifact_type classification   n
        checkpoint         legacy   5
        checkpoint    provisional  51
        checkpoint       reusable   2
config_or_document  authoritative  26
config_or_document      generated   7
          contract  authoritative   2
          notebook    provisional  31
          notebook       reusable  15
 output_collection      generated   8
 output_collection         legacy   4
 output_collection    provisional   4
            report  authoritative   3
            report      generated 222
            report         legacy  25
inventory rows=405, quarantined=34


## Success criterion

Every recorded artifact has one allowed classification, paths are unique, raw data filenames are absent and all legacy or smoke-only artifacts are non-selectable through the foundation configs.